# Xhakinha Clips AI — sessão Kaggle
Ativar: Settings → Internet **On**; Accelerator **GPU T4/P100** (opcional); Add-ons → Secrets (lista em KAGGLE_SETUP.txt).

In [ ]:
import os, sys, subprocess, base64
from pathlib import Path
from kaggle_secrets import UserSecretsClient

# Credencial protegida no Kaggle Secrets
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
if not token:
    raise RuntimeError("GITHUB_TOKEN não encontrado nos Secrets do Kaggle.")

# Autenticação temporária: não grava o token no repositório
auth = base64.b64encode(
    f"x-access-token:{token}".encode()
).decode()

env = os.environ.copy()
env.update({
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
})

REPO = "https://github.com/gav74742-star/Xhakinha.git"
APP = "/kaggle/working/app"
PROJECT = os.path.join(APP, "xhakinha-clips-ai")

if not os.path.isdir(os.path.join(APP, ".git")):
    if os.path.exists(APP):
        raise RuntimeError(
            "A pasta app existe, mas não é um clone Git. "
            "Não será apagada automaticamente."
        )
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO, APP],
        env=env, check=True
    )
else:
    subprocess.run(
        ["git", "-C", APP, "pull", "--ff-only"],
        env=env, check=True
    )

requirements = os.path.join(PROJECT, "requirements-kaggle.txt")
if not os.path.isfile(requirements):
    raise FileNotFoundError(requirements)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", requirements],
    check=True
)

sys.path.insert(0, PROJECT)
print("Clone e instalação concluídos.")


In [ ]:
from xhakinha.cloud.healthcheck import local_report
import json; print(json.dumps(local_report(), indent=2))

In [ ]:
# Roda até ~10,5 h (cérebro + executor). Pode interromper a qualquer momento: nada fica só aqui.
from xhakinha.cloud import kaggle_runner
kaggle_runner.main()